# Tune SciBERT + LoRA

Copied from `3_train/train_lora.ipynb`; Optuna replaces only its training loop. This notebook uses train and validation only—never test.


## 1. Imports
Keep dependencies here and experiment settings in the next section.

In [19]:
import gc
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn.functional as F
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, set_seed


## 2. Configuration
Choose `PAIR_READER` to switch between random and topic-matched SPECTER pairs. SPECTER mode removes detected duplicate works before splitting, samples distinct positive works, and matches negative similarity to each positive.

In [20]:
ROOT = Path.cwd().resolve()
if ROOT.name == '6_tuning': ROOT = ROOT.parent
DATA_PATH = Path('/mnt/research/deepthought/SciAR/SciAR/data/sciar_refereed_v1/papers.parquet')
READER_PATH = ROOT / '2_Data/data_reader.ipynb'
OUTPUT_DIR = ROOT / '6_tuning/outputs/scibert_quick'
PAIR_READER = 'specter'
SPECTER_MODEL = 'allenai/specter'; SPECTER_CACHE_DIR = ROOT / 'data/cache/specter'
SPECTER_DEVICES = [f'cuda:{i}' for i in range(torch.cuda.device_count())] or ['cpu']
SPECTER_PRECISION = 'float16' if torch.cuda.is_available() else 'float32'
SPECTER_BATCH_SIZE = 32; SPECTER_MATCH_BATCH_SIZE = 16
SPECTER_PAPER_LIMIT = 2; SPECTER_AUTHOR_LIMIT = 10
LORA_R = 8; LORA_ALPHA = 16; LORA_DROPOUT = 0.1
MODEL_NAME = 'allenai/scibert_scivocab_uncased'; SEED = 67; MAX_PAPERS = None

TRAIN_PAIRS = 20_000; VAL_PAIRS = 2_000; MAX_LENGTH = 512; BATCH_SIZE = 64; EPOCHS = 3; N_TRIALS = 25
set_seed(SEED); DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'; USE_AMP = USE_CUDA
assert DATA_PATH.is_file(), f'Dataset not found: {DATA_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE} | Output: {OUTPUT_DIR}')
if torch.cuda.is_available():
  device_names = [
      torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())
  ]
  for idx, name in enumerate(device_names):
    print(f"cuda:{idx} -> {name}")
else:
  print("CUDA is not available.")


Device: cuda | Output: /mnt/ufs18/rs/deepthought/SciAR/SciAR/6_tuning/outputs/scibert_quick
cuda:0 -> NVIDIA H200
cuda:1 -> NVIDIA H200
cuda:2 -> NVIDIA H200
cuda:3 -> NVIDIA H200


## 3. Load papers and split authors
Use the reader's 80/10/10 author split before generating pairs. Single-paper authors can contribute negative pairs. Load only the tagged reader cells so its demo does not run. Save author assignments for reproducibility.

In [21]:
reader_cells = json.loads(READER_PATH.read_text())['cells']
reader = {}
for cell in reader_cells:
    if cell['cell_type'] == 'code' and 'reader-code' in cell.get('metadata', {}).get('tags', []):
        exec(''.join(cell['source']), reader)
load_papers = reader['load_papers']
split_authors = reader['split_authors']
make_pairs = reader['make_pairs']

all_papers = load_papers(DATA_PATH, max_papers=MAX_PAPERS)
papers = all_papers
if PAIR_READER == 'specter':
    import importlib.util
    spec = importlib.util.spec_from_file_location('specter_reader', ROOT / '2_Data/specter_reader.py')
    specter_reader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(specter_reader)
    retained_indices, dedup_stats = specter_reader.deduplicate_works(all_papers)
    papers = all_papers.iloc[retained_indices].reset_index(drop=True)
    (OUTPUT_DIR / 'dedup_stats.json').write_text(json.dumps(dedup_stats, indent=2))
    print('Duplicate-work filter:', dedup_stats)
assert papers['bibcode'].is_unique, 'Expected one row per bibcode.'
author_splits = split_authors(papers, seed=SEED)
assert all(len(ids) >= 2 for ids in author_splits.values()), 'Need at least two authors per split.'
assert not (set(author_splits['train']) & set(author_splits['validation'])
            or set(author_splits['train']) & set(author_splits['test'])
            or set(author_splits['validation']) & set(author_splits['test']))
paper_groups = papers.groupby('first_author_orcid').indices
(OUTPUT_DIR / 'author_splits.json').write_text(json.dumps(author_splits, indent=2))
for name, ids in author_splits.items():
    print(f'{name}: {len(ids):,} authors, {sum(len(paper_groups[a]) for a in ids):,} papers')


Duplicate-work filter: {'input_papers': 679721, 'retained_papers': 672677, 'conflicting_author_papers_removed': 1139, 'duplicate_papers_removed': 5905}
train: 178,295 authors, 538,127 papers
validation: 22,286 authors, 67,383 papers
test: 22,288 authors, 67,167 papers


## 4. Sample balanced pairs
Create balanced pairs. SPECTER mode samples distinct positives and matches each to a different-author negative with similar SPECTER similarity. Random sampling can repeat pairs.

In [22]:
if PAIR_READER == 'specter':
    specter_vectors = specter_reader.specter_embeddings(
        all_papers, SPECTER_CACHE_DIR, model_name=SPECTER_MODEL,
        batch_size=SPECTER_BATCH_SIZE, devices=SPECTER_DEVICES, precision=SPECTER_PRECISION)
    specter_vectors = specter_vectors[retained_indices]
pair_sets = {}
for offset, (name, size) in enumerate(
        [('train', TRAIN_PAIRS), ('validation', VAL_PAIRS)]):
    if PAIR_READER == 'specter':
        pair_sets[name] = specter_reader.make_matched_pairs(
            papers, specter_vectors, author_splits[name], size,
            paper_limit=SPECTER_PAPER_LIMIT, author_limit=SPECTER_AUTHOR_LIMIT,
            batch_size=SPECTER_MATCH_BATCH_SIZE, device=DEVICE, seed=SEED + offset)
    else:
        pair_sets[name] = make_pairs(papers, paper_groups, author_splits[name], size, SEED + offset)
for name, examples in pair_sets.items():
    assert examples['label'].value_counts().to_dict() == {0: len(examples) // 2, 1: len(examples) // 2}
    columns = ['bibcode_a', 'bibcode_b', 'label']
    if 'specter_similarity' in examples: columns.append('specter_similarity')
    if 'match_id' in examples: columns.append('match_id')
    examples[columns].to_csv(OUTPUT_DIR / f'{name}_pairs.csv', index=False)
    print(f'{name}: {len(examples):,} pairs')

(OUTPUT_DIR / 'author_splits.json').write_text(json.dumps(author_splits, indent=2))


Using SPECTER cache: /mnt/ufs18/rs/deepthought/SciAR/SciAR/data/cache/specter/ecd7ba273e058bb1050f392d845bcb993e8152453673ff549e911eb884afa789.npy


KeyboardInterrupt: 

## 5. Tokenizer and LoRA encoder
Tokenize each reader text (title, blank line, abstract) on its own. Both papers share the same encoder. Only LoRA query/value adapters train; the base BERT weights stay frozen. Masked mean pooling produces a normalized embedding without using BERT's pooler.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False)
model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['query', 'value'], bias='none',
)).to(DEVICE)
model.print_trainable_parameters()

def collate_pairs(batch):
    # first half is A, second half is B; encode each paper separately
    texts = [row['text_a'] for row in batch] + [row['text_b'] for row in batch]
    tokens = tokenizer(
        texts,
        padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt',
    )
    # CosineEmbeddingLoss expects -1 for different authors, while saved labels stay 0/1
    labels = torch.tensor([1 if row['label'] == 1 else -1 for row in batch], dtype=torch.float32)
    return tokens, labels

loaders = {
    name: DataLoader(
        examples.to_dict('records'), batch_size=BATCH_SIZE, shuffle=(name == 'train'),
        collate_fn=collate_pairs, num_workers=0, pin_memory=USE_CUDA,
        generator=torch.Generator().manual_seed(SEED),
    )
    for name, examples in pair_sets.items()
}

def encode(tokens):
    tokens = {key: value.to(DEVICE, non_blocking=USE_CUDA) for key, value in tokens.items()}
    hidden = model(**tokens).last_hidden_state
    mask = tokens['attention_mask'].unsqueeze(-1).to(hidden.dtype)
    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return F.normalize(pooled, p=2, dim=-1)


## 6. Loss and evaluation
Use `CosineEmbeddingLoss(margin=0.2)`: **same author:** `1 − cosine_similarity`; **different author:** `max(0, cosine_similarity − margin)`. CUDA forward passes use float16 autocast and gradient scaling; the loss uses float32 embeddings. Validation ROC-AUC selects the best epoch; a validation-only threshold maximizes balanced accuracy (equivalent to accuracy for these balanced pairs).

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []

    for tokens, labels in tqdm(loader, desc="Evaluate", leave=False):
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            left, right = encode(tokens).chunk(2)

        loss = criterion(
            left.float(),
            right.float(),
            labels.to(DEVICE, non_blocking=USE_CUDA),
        )
        total_loss += loss.item() * len(labels)
        all_labels.extend((labels > 0).int().tolist())
        all_scores.extend(
            (left.float() * right.float()).sum(dim=-1).cpu().tolist()
        )

    return total_loss / len(loader.dataset), np.array(all_labels), np.array(all_scores)


## 7. Optuna training

Each trial uses fresh SciBERT weights and the same training/validation logic as `3_train`.


In [ ]:
def objective(trial):
    global model, criterion, scaler, optimizer
    set_seed(SEED)
    r = trial.suggest_categorical('lora_r', [4, 8, 16, 32])
    multiplier = trial.suggest_categorical('alpha_multiplier', [1, 2, 4])
    lr = trial.suggest_float('learning_rate', 1e-5, 5e-4, log=True)
    wd = trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True)
    margin = trial.suggest_float('margin', 0.0, 0.5)
    dropout = trial.suggest_float('lora_dropout', 0.0, 0.2)
    base_model = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False)
    model = get_peft_model(base_model, LoraConfig(task_type=TaskType.FEATURE_EXTRACTION,
        r=r, lora_alpha=r * multiplier, lora_dropout=dropout,
        target_modules=['query', 'value'], bias='none')).to(DEVICE)
    criterion = torch.nn.CosineEmbeddingLoss(margin=margin)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=wd)
    best_auc, history = -float('inf'), []
    try:
        for epoch in range(1, EPOCHS + 1):
            model.train(); total_loss = 0.0
            for tokens, labels in tqdm(loaders['train'], desc=f'Trial {trial.number}, epoch {epoch}/{EPOCHS}'):
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                    left, right = encode(tokens).chunk(2)
                loss = criterion(left.float(), right.float(), labels.to(DEVICE, non_blocking=USE_CUDA))
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer); scaler.update(); total_loss += loss.item() * len(labels)
            val_loss, val_labels, val_scores = evaluate(loaders['validation'])
            val_auc = float(roc_auc_score(val_labels, val_scores))
            history.append(dict(epoch=epoch, train_loss=total_loss / TRAIN_PAIRS, val_loss=val_loss, val_auc=val_auc))
            best_auc = max(best_auc, val_auc); trial.report(best_auc, step=epoch)
            if trial.should_prune(): raise optuna.TrialPruned()
        trial.set_user_attr('history', history)
        return best_auc
    finally:
        del model; gc.collect()
        if USE_CUDA: torch.cuda.empty_cache()
storage = f"sqlite:///{(OUTPUT_DIR / 'study.db').resolve()}"
study = optuna.create_study(study_name='scibert_quick', storage=storage, load_if_exists=True,
    direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)


[I 2026-11-18 14:55:08,956] Using an existing study with `study_name='scibert_quick'` instead of creating a new one.
/tmp/ipykernel_2536812/1845922332.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
Trial 1, epoch 1/3:   6%|▌         | 153/2500 [00:09<02:19, 16.77it/s]
[W 2026-11-18 14:55:19,347] Trial 1 failed with parameters: {'lora_r': 8, 'alpha_multiplier': 2, 'learning_rate': 0.00038397312410446094, 'weight_decay': 0.014738554665157583, 'margin': 0.0234476533341898, 'lora_dropout': 0.16426216987689957} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/mnt/ffs24/home/petrecc1/miniforge3/envs/SciARvenv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 210, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_2536812/1845922332.py", line

KeyboardInterrupt: 

## 8. Best validation configuration

Use this in a fresh full `3_train` run, then test once.


In [ ]:
summary = {'best_validation_auc': study.best_value, 'best_params': study.best_params,
           'completed_trials': len(study.trials), 'note': 'No test pairs were created, loaded, or scored.'}
(OUTPUT_DIR / 'best_trial.json').write_text(json.dumps(summary, indent=2))
display(pd.DataFrame([{'number': t.number, 'state': t.state.name, 'validation_auc': t.value, **t.params} for t in study.trials]).sort_values('validation_auc', ascending=False))
print(json.dumps(summary, indent=2))
